In [ ]:
# Semantic Segmentation with ResNet on Pascal VOC Dataset
# Refined script for training a ResNet-based segmentation model.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.v2 as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os

# =============================================================================
# 1. Configuration & Constants
# =============================================================================

class CFG:
    """Configuration class for hyperparameters and settings."""
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    DATA_ROOT = './data'
    MODEL_PATH = 'resnet_segmentation_model.pth'
    NUM_CLASSES = 21  # 20 classes + 1 background
    IGNORE_INDEX = 255
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 16
    NUM_EPOCHS = 25  # A more reasonable number for demonstration
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    SCHEDULER_STEP = 10
    SCHEDULER_GAMMA = 0.1
    NUM_WORKERS = os.cpu_count() // 2

# PASCAL VOC color palette for visualization
# Provides a unique color for each class.
PALETTE = np.array([
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
    [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0],
    [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128],
    [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
    [0, 64, 128]
], dtype=np.uint8)

# =============================================================================
# 2. Data Handling & Visualization Utilities
# =============================================================================

def get_datasets():
    """Prepares and returns the training and validation datasets."""
    # The target mask is a PIL Image. This transform converts it to a tensor of class indices.
    target_transform = transforms.Compose([
        transforms.Resize(CFG.IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
        transforms.Lambda(lambda x: torch.from_numpy(np.array(x)).long())
    ])

    # Standard transforms for the input image.
    image_transform = transforms.Compose([
        transforms.Resize(CFG.IMG_SIZE),
        transforms.ToDtype(torch.float32, scale=True), # Converts to float and scales to [0, 1]
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # The same transforms are applied to both image and target in VOCSegmentation
    train_dataset = torchvision.datasets.VOCSegmentation(
        root=CFG.DATA_ROOT, year='2012', image_set='train', download=True,
        transform=image_transform, target_transform=target_transform
    )

    val_dataset = torchvision.datasets.VOCSegmentation(
        root=CFG.DATA_ROOT, year='2012', image_set='val', download=True,
        transform=image_transform, target_transform=target_transform
    )
    return train_dataset, val_dataset

def denormalize(tensor):
    """Denormalizes a tensor image for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return torch.clamp(tensor, 0, 1)

def colorize_mask(mask, palette):
    """Applies a color palette to a segmentation mask."""
    # Set pixels with ignore_index to 0 (background) for visualization
    mask_np = mask.cpu().numpy().astype(np.uint8)
    mask_np[mask_np == CFG.IGNORE_INDEX] = 0
    return palette[mask_np]

def visualize_sample(image, ground_truth, prediction=None):
    """Visualizes the original image, ground truth, and optional model prediction."""
    image_vis = denormalize(image).permute(1, 2, 0)
    gt_vis = colorize_mask(ground_truth, PALETTE)

    num_plots = 2 if prediction is None else 3
    plt.figure(figsize=(5 * num_plots, 5))

    plt.subplot(1, num_plots, 1)
    plt.imshow(image_vis)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, num_plots, 2)
    plt.imshow(gt_vis)
    plt.title('Ground Truth')
    plt.axis('off')

    if prediction is not None:
        pred_vis = colorize_mask(prediction, PALETTE)
        plt.subplot(1, num_plots, 3)
        plt.imshow(pred_vis)
        plt.title('Prediction')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# =============================================================================
# 3. Model Definition
# =============================================================================

class DecoderBlock(nn.Module):
    """A single block for the decoder, performing upsampling."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class ResNetSegmentation(nn.Module):
    """A simple semantic segmentation model using a ResNet18 backbone."""
    def __init__(self, num_classes):
        super().__init__()
        # Use modern `weights` API for pretrained models
        resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # Encoder (ResNet backbone without final layers)
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])

        # Decoder (upsampling layers to restore original resolution)
        self.decoder = nn.Sequential(
            DecoderBlock(512, 256),
            DecoderBlock(256, 128),
            DecoderBlock(128, 64),
            DecoderBlock(64, 32),
            nn.Conv2d(32, num_classes, kernel_size=1) # Final classification layer
        )
        # Bilinear upsampling to match the input size perfectly
        self.final_upsample = nn.Upsample(size=CFG.IMG_SIZE, mode='bilinear', align_corners=False)

    def forward(self, x):
        features = self.encoder(x)
        logits = self.decoder(features)
        output = self.final_upsample(logits)
        return output

# =============================================================================
# 4. Evaluation Metrics
# =============================================================================

def calculate_metrics(preds, targets, num_classes):
    """Calculates pixel accuracy and mean IoU."""
    # Create a mask to ignore specified index
    mask = targets != CFG.IGNORE_INDEX
    
    # Pixel Accuracy
    correct_pixels = torch.sum((preds[mask] == targets[mask]))
    total_pixels = torch.sum(mask)
    pixel_acc = correct_pixels.float() / total_pixels if total_pixels > 0 else 0.0

    # Mean Intersection over Union (mIoU)
    iou_per_class = []
    for cls in range(num_classes):
        pred_inds = (preds == cls) & mask
        target_inds = (targets == cls) & mask
        
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        
        if union == 0:
            # If there's no ground truth or prediction for this class, IoU is not well-defined.
            # Some implementations assign 1.0 if the class is correctly not present.
            # We will skip it for a more conservative estimate.
            continue
        
        iou_per_class.append(intersection / union)

    miou = np.mean(iou_per_class) if iou_per_class else 0.0
    
    return pixel_acc.item(), miou

# =============================================================================
# 5. Training & Validation Loops
# =============================================================================

def train_one_epoch(model, dataloader, criterion, optimizer):
    """Runs a single training epoch."""
    model.train()
    total_loss = 0.0
    
    for images, targets in tqdm(dataloader, desc="Training"):
        images, targets = images.to(CFG.DEVICE), targets.to(CFG.DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    """Evaluates the model on the validation set."""
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    
    for images, targets in tqdm(dataloader, desc="Evaluating"):
        images, targets = images.to(CFG.DEVICE), targets.to(CFG.DEVICE)
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds)
        all_targets.append(targets)
    
    # Concatenate all batches
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    
    avg_loss = total_loss / len(dataloader)
    pixel_acc, miou = calculate_metrics(all_preds, all_targets, CFG.NUM_CLASSES)
    
    return avg_loss, pixel_acc, miou

# =============================================================================
# 6. Main Execution Block
# =============================================================================

def main():
    """Main function to run the training and evaluation pipeline."""
    print(f"Using device: {CFG.DEVICE}")

    # --- Data Loading ---
    train_dataset, val_dataset = get_datasets()
    train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
    print(f"Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

    # --- Visualize a Sample ---
    print("\nVisualizing a data sample:")
    img_sample, mask_sample = train_dataset[0]
    visualize_sample(img_sample, mask_sample)
    
    # --- Model, Loss, Optimizer ---
    model = ResNetSegmentation(num_classes=CFG.NUM_CLASSES).to(CFG.DEVICE)
    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")
    
    criterion = nn.CrossEntropyLoss(ignore_index=CFG.IGNORE_INDEX)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=CFG.SCHEDULER_STEP, gamma=CFG.SCHEDULER_GAMMA)
    
    # --- Training Loop ---
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_miou': []}
    best_miou = 0.0

    print(f"\nStarting training for {CFG.NUM_EPOCHS} epochs...")
    for epoch in range(CFG.NUM_EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{CFG.NUM_EPOCHS} ---")
        
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_miou = evaluate(model, val_loader, criterion)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_miou'].append(val_miou)
        
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Pixel Acc: {val_acc:.4f} | Val mIoU: {val_miou:.4f}")

        # Save the best model based on mIoU
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), CFG.MODEL_PATH)
            print(f"🚀 New best model saved to {CFG.MODEL_PATH} with mIoU: {best_miou:.4f}")

    print("\n✅ Training complete!")

    # --- Final Results & Visualization ---
    # Load best model for final visualization
    model.load_state_dict(torch.load(CFG.MODEL_PATH))
    
    print("\nVisualizing predictions with the best model:")
    model.eval()
    for i in range(3):
        image, target = val_dataset[i]
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(CFG.DEVICE))
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu()
        visualize_sample(image, target, prediction)

if __name__ == '__main__':
    main()